<a href="https://colab.research.google.com/github/dialejobv/ejemplo_programacion/blob/main/Ejem_MySQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🛠️ Configuración Inicial en Google Colab**

In [24]:
# ============================================================
# PASO 1 CORREGIDO: Instalar, arrancar MySQL y crear usuario
# ============================================================

!apt-get update -qq
!apt-get install mysql-server -y -qq
!pip install mysql-connector-python -q

# Arrancar el servicio
!service mysql start
!service mysql status

# Crear un usuario con contraseña para la conexión desde Python
# Ejecutamos el comando como el usuario del sistema "root" para saltar el auth_socket
!sudo mysql -u root -e "CREATE USER IF NOT EXISTS 'colab_user'@'localhost' IDENTIFIED BY 'colab_pass';"
!sudo mysql -u root -e "GRANT ALL PRIVILEGES ON *.* TO 'colab_user'@'localhost' WITH GRANT OPTION;"
!sudo mysql -u root -e "FLUSH PRIVILEGES;"

print("✅ Servidor MySQL listo y usuario 'colab_user' creado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
 * /usr/bin/mysqladmin  Ver 8.0.46-0ubuntu0.24.04.4 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2026, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.46-0ubuntu0.24.04.4
Protocol version	10
Connection		Localhost via UNIX socket
UNIX socket		/var/run/mysqld/mysqld.sock
Uptime:			1 min 21 sec

Threads: 2  Questions: 10  Slow queries: 0  Opens: 119  Flush tables: 3  Open tables: 38  Queries per second avg: 0.123
✅ Servidor MySQL listo y usuario 'colab_user' creado


**Explicación:**


*   **apt-get install mysql-server** instala el motor de base de datos dentro del contenedor de Colab

*   **pip install mysql-connector-python** instala el "puente" entre Python y MySQL.

*   **!sudo mysql -u root -e "...":** El uso de sudo es la clave. Permite que el cliente de MySQL se conecte localmente como el usuario root del sistema, lo que satisface la autenticación por auth_socket . El flag -e ejecuta un comando directamente.

*  **CREATE USER 'colab_user'@'localhost' IDENTIFIED BY 'colab_pass':** Crea un nuevo usuario que sí se autentica por contraseña

*  **GRANT ALL PRIVILEGES ON *.* ...:** Le da todos los permisos para que puedas trabajar sin restricciones.



# **🐍 Conexión y Creación de Base de Datos (Python)**

In [25]:
print('Verificando el estado del servidor MySQL...')
!ps aux | grep mysqld

Verificando el estado del servidor MySQL...
mysql      10790  0.0  0.0   2804  2028 ?        S    10:39   0:00 /bin/sh /usr/bin/mysqld_safe
mysql      10937  2.4  2.9 1787556 396956 ?      Sl   10:39   0:02 /usr/sbin/mysqld --basedir=/usr --datadir=/var/lib/mysql --plugin-dir=/usr/lib/mysql/plugin --log-error=/var/log/mysql/error.log --pid-file=8be59c90dc54.pid
root       11846  0.0  0.0   7344  3532 ?        S    10:41   0:00 /bin/bash -c ps aux | grep mysqld
root       11848  0.0  0.0   6548  2304 ?        S    10:41   0:00 grep mysqld


In [26]:
print('Verificando si el puerto 3306 de MySQL está abierto y escuchando...')
!netstat -tulnp | grep 3306

Verificando si el puerto 3306 de MySQL está abierto y escuchando...
tcp        0      0 127.0.0.1:3306          0.0.0.0:*               LISTEN      10937/mysqld        
tcp        0      0 127.0.0.1:33060         0.0.0.0:*               LISTEN      10937/mysqld        


In [27]:
!/etc/init.d/mysql restart

[ OK ]
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]


In [28]:
# ============================================================
# PASO 2: Conectar a MySQL con el usuario creado
# ============================================================
import mysql.connector

cnx = mysql.connector.connect(
    host="127.0.0.1",
    user="colab_user",      # Usuario creado
    password="colab_pass"   # Contraseña del usuario
)

cursor = cnx.cursor(buffered=True)
cursor.execute("CREATE DATABASE IF NOT EXISTS tienda_db")
cursor.execute("USE tienda_db")

print("✅ Base de datos 'tienda_db' lista")

✅ Base de datos 'tienda_db' lista


**Explicación:**

*  El cursor es como un "puntero" que recorre los resultados y ejecuta sentencias.

*  **USE tienda_db** indica al servidor sobre qué base de datos trabajar.

# **PASO 3: Crear tablas con relaciones (FOREIGN KEY)**

**Crear una tabla**

Puedes definir la estructura de tus datos con CREATE TABLE. Es común usar AUTO_INCREMENT para generar IDs automáticamente

In [29]:
# ============================================================
# PASO 3: Crear tablas con relaciones
# ============================================================

# 3.1 Tabla de clientes
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id INT PRIMARY KEY AUTO_INCREMENT,
    nombre VARCHAR(64) NOT NULL,
    ciudad VARCHAR(64)
)
""")

# 3.2 Tabla de productos
cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    id INT PRIMARY KEY AUTO_INCREMENT,
    nombre VARCHAR(64) NOT NULL,
    precio DECIMAL(10,2) NOT NULL
)
""")

# 3.3 Tabla de pedidos: conecta clientes con productos
#     FOREIGN KEY garantiza integridad referencial
cursor.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    id INT PRIMARY KEY AUTO_INCREMENT,
    cliente_id INT,
    producto_id INT,
    cantidad INT DEFAULT 1,
    fecha DATE,
    FOREIGN KEY (cliente_id) REFERENCES clientes(id),
    FOREIGN KEY (producto_id) REFERENCES productos(id)
)
""")

print("✅ Tablas creadas: clientes, productos, pedidos")

✅ Tablas creadas: clientes, productos, pedidos


**Explicación:**

*    **AUTO_INCREMENT** genera IDs automáticamente (1, 2, 3…).

*   **DECIMAL(10,2)** es ideal para precios (evita errores de redondeo de FLOAT).

*   **FOREIGN KEY** asegura que un pedido no pueda apuntar a un cliente o producto inexistente. Es la base de la integridad referencial.



**🔹 PASO 4: Insertar datos**

In [30]:
# ============================================================
# PASO 4: Insertar datos de ejemplo
# ============================================================

# 4.1 Insertar clientes (executemany para varios registros a la vez)
cursor.executemany(
    "INSERT INTO clientes (nombre, ciudad) VALUES (%s, %s)",
    [("Ana", "Bogotá"), ("Luis", "Medellín"), ("Marta", "Cali")]
)

# 4.2 Insertar productos
cursor.executemany(
    "INSERT INTO productos (nombre, precio) VALUES (%s, %s)",
    [("Laptop", 1200.00), ("Mouse", 25.50), ("Teclado", 45.00)]
)

# 4.3 Insertar pedidos
cursor.executemany(
    "INSERT INTO pedidos (cliente_id, producto_id, cantidad, fecha) VALUES (%s, %s, %s, %s)",
    [
        (1, 1, 1, "2024-01-15"),  # Ana compra 1 Laptop
        (1, 2, 2, "2024-01-20"),  # Ana compra 2 Mouses
        (2, 3, 1, "2024-02-05"),  # Luis compra 1 Teclado
        (3, 1, 1, "2024-02-10"),  # Marta compra 1 Laptop
        (2, 2, 3, "2024-02-15"),  # Luis compra 3 Mouses
    ]
)

# 4.4 IMPORTANTE: confirmar los cambios
cnx.commit()
print("✅ Datos insertados correctamente")

✅ Datos insertados correctamente


**tExplicación:**

*   **%s** son placeholders: evitan inyección SQL y escapes manuales.

*  **executemany()** es mucho más eficiente que ejecutar varios INSERT sueltos.

*  **cnx.commit() confirma la transacción. Sin esto, los datos no se guardan permanentemente.**

**🔹 PASO 5: Consultas con JOIN, GROUP BY y funciones de agregación**

In [31]:
# ============================================================
# PASO 5: Consultas avanzadas
# ============================================================

# 5.1 JOIN: ver el detalle de cada pedido con nombre de cliente y producto
query_join = """
SELECT p.id, c.nombre AS cliente, pr.nombre AS producto,
       p.cantidad, pr.precio,
       (p.cantidad * pr.precio) AS total
FROM pedidos p
JOIN clientes c  ON p.cliente_id = c.id
JOIN productos pr ON p.producto_id = pr.id
ORDER BY p.id
"""
cursor.execute(query_join)

print("📋 DETALLE DE PEDIDOS")
print("-" * 70)
for fila in cursor:
    print(f"Pedido #{fila[0]} | {fila[1]:<8} | {fila[2]:<10} | "
          f"Cant: {fila[3]} | Total: ${fila[5]:.2f}")

# 5.2 GROUP BY: total gastado por cliente
query_total = """
SELECT c.nombre, SUM(p.cantidad * pr.precio) AS total_gastado
FROM pedidos p
JOIN clientes c  ON p.cliente_id = c.id
JOIN productos pr ON p.producto_id = pr.id
GROUP BY c.nombre
ORDER BY total_gastado DESC
"""
cursor.execute(query_total)

print("\n💰 TOTAL GASTADO POR CLIENTE")
print("-" * 70)
for nombre, total in cursor:
    print(f"{nombre:<10} → ${total:.2f}")

📋 DETALLE DE PEDIDOS
----------------------------------------------------------------------
Pedido #1 | Ana      | Laptop     | Cant: 1 | Total: $1200.00
Pedido #2 | Ana      | Mouse      | Cant: 2 | Total: $51.00
Pedido #3 | Luis     | Teclado    | Cant: 1 | Total: $45.00
Pedido #4 | Marta    | Laptop     | Cant: 1 | Total: $1200.00
Pedido #5 | Luis     | Mouse      | Cant: 3 | Total: $76.50

💰 TOTAL GASTADO POR CLIENTE
----------------------------------------------------------------------
Ana        → $1251.00
Marta      → $1200.00
Luis       → $121.50


**Explicación:**

*  JOIN combina filas de varias tablas usando una columna común (cliente_id, producto_id).

*  GROUP BY agrupa filas para aplicar funciones de agregación (SUM, COUNT, AVG).

*  AS crea alias para que el resultado sea más legible.

**🔹 PASO 6: Análisis con Pandas (integración)**

In [32]:
# ============================================================
# PASO 6: Llevar los resultados a un DataFrame de pandas
# ============================================================
import pandas as pd

query_df = """
SELECT c.nombre AS cliente, c.ciudad,
       COUNT(p.id) AS num_pedidos,
       SUM(p.cantidad * pr.precio) AS total_gastado
FROM pedidos p
JOIN clientes c  ON p.cliente_id = c.id
JOIN productos pr ON p.producto_id = pr.id
GROUP BY c.nombre, c.ciudad
"""

# read_sql combina la consulta + conexión en un DataFrame
df = pd.read_sql(query_df, cnx)
print(df)

# Estadísticas rápidas
print("\n📊 Estadísticas:")
print(df[["num_pedidos", "total_gastado"]].describe())

  cliente    ciudad  num_pedidos  total_gastado
0     Ana    Bogotá            2         1251.0
1    Luis  Medellín            2          121.5
2   Marta      Cali            1         1200.0

📊 Estadísticas:
       num_pedidos  total_gastado
count     3.000000       3.000000
mean      1.666667     857.500000
std       0.577350     637.904578
min       1.000000     121.500000
25%       1.500000     660.750000
50%       2.000000    1200.000000
75%       2.000000    1225.500000
max       2.000000    1251.000000


/tmp/ipykernel_4019/1453371026.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_df, cnx)


**Explicación:**

*  pd.read_sql() es el puente ideal: SQL trae los datos, pandas los analiza.

*  A partir de aquí puedes graficar con matplotlib, hacer correlaciones, etc.

**🔹 PASO 7: Cierre limpio**

In [33]:
# ============================================================
# PASO 7: Cerrar cursor y conexión
# ============================================================
cursor.close()
cnx.close()
print("🔒 Conexión cerrada correctamente")

🔒 Conexión cerrada correctamente


**Explicación:**

*  Liberar recursos evita conexiones "zombie" que consumen memoria en el servidor.